# Project goal

The final objective of this project is to design a battery dispatch / MPC system
that can operate under uncertainty in renewable generation and load.

To reach this goal the project is divided into phases:

Phase 1 — Build data  
Phase 2 — Understand data  
Phase 3 — Model system  
Phase 4 — MPC

This notebook corresponds to Phase 1.

## Phase 1 — Data Pipeline and Validation

### Goal: Ensure that the data used later for uncertainty analysis and battery dispatch is physically and statistically meaningful.                         

### Data Sources

We use ENTSO-E based data for:
- Load forecast
- Wind and solar forecast
- Load actual
- Solar actual
- Wind onshore actual
- Wind offshore actual

The project initially attempted to use a generic generation forecast endpoint,
but validation showed that this returned aggregated generation forecasts rather than
technology-specific forecasts.
This was corrected by switching to the renewable-specific forecast endpoint.

### Problems encountered during pipeline construction

Several issues had to be resolved before error analysis was meaningful:

1. Mixed timezones in timestamps
2. Mixed time resolutions (15 min / 30 min / hourly)
3. Incorrect renewable forecast extraction
4. Forecast files with duplicated values across technologies
5. Need for consistent naming and merge logic

### Why hourly resolution was adopted

The raw data appears in mixed time resolutions, especially 15-minute actuals and forecast products that may not align perfectly.

Since the later dispatch model and market interpretation are naturally hourly,
all series were resampled to hourly resolution before merging.

This choice avoids artificial comparison between mismatched timestamps
and produces a coherent dataset for later control design.

In [1]:
import pandas as pd

solar_actual = pd.read_csv("../data/actual/DE_solar_actuals_2023.csv")
solar_actual.head()

,utc_timestamp,DE_solar_actual
0,2023-01-01 00:00:00+01:00,2.28
1,2023-01-01 00:15:00+01:00,1.73
2,2023-01-01 00:30:00+01:00,1.43
3,2023-01-01 00:45:00+01:00,1.73
4,2023-01-01 01:00:00+01:00,1.51


### Forecast validation

Before computing forecast errors, the forecast data had to be checked for physical consistency.

Simple sanity checks were applied to ensure that the forecast values correspond to realistic power system behavior.

The following conditions were verified:

- Solar generation should be near zero during nighttime hours.
- Wind onshore and wind offshore forecasts should be different time series, not duplicated values.
- Load forecasts should be in a realistic magnitude range for Germany and should follow a daily pattern.

These checks were necessary because an earlier version of the pipeline used a generic generation forecast endpoint,
which returned aggregated values instead of technology-specific forecasts.

Using incorrect forecast data would make the later error analysis meaningless,
therefore forecast validation was performed before building the merged dataset.

In [2]:
merged = pd.read_csv("../data/processed/DE_merged_2025.csv")
merged.head()

,utc_timestamp,DE_load_actual,DE_solar_actual,DE_wind_onshore_actual,DE_wind_offshore_actual,DE_load_forecast,DE_solar_forecast,DE_wind_onshore_forecast,DE_wind_offshore_forecast,solar_error,wind_onshore_error,wind_offshore_error,load_error
0,2025-01-01 00:00:00+01:00,47741.7825,9.5250,32875.6525,2502.1700,44518.2425,0.0,38472.2225,3759.6575,-9.5250,5596.5700,1257.4875,-3223.5400
1,2025-01-01 01:00:00+01:00,46867.3225,9.5150,33519.9725,2395.2675,42269.3500,0.0,38931.1575,3795.0700,-9.5150,5411.1850,1399.8025,-4597.9725
2,2025-01-01 02:00:00+01:00,45797.5125,9.2650,33767.0825,2614.1375,41095.9975,0.0,39388.8900,3826.1900,-9.2650,5621.8075,1212.0525,-4701.5150
3,2025-01-01 03:00:00+01:00,44623.4550,9.3875,32876.2300,1684.2200,40805.1225,0.0,39347.5000,3336.8875,-9.3875,6471.2700,1652.6675,-3818.3325
4,2025-01-01 04:00:00+01:00,43626.0875,9.1900,33113.9450,1327.7375,40678.0000,0.0,39681.4475,3213.5925,-9.1900,6567.5025,1885.8550,-2948.0875


### Structure of merged dataset

After preprocessing, all time series were merged into a single hourly dataset.

Each merged file contains the following groups of columns:

Actual values:
- DE_load_actual
- DE_solar_actual
- DE_wind_onshore_actual
- DE_wind_offshore_actual

Forecast values:
- DE_load_forecast
- DE_solar_forecast
- DE_wind_onshore_forecast
- DE_wind_offshore_forecast

Error values (actual − forecast):
- load_error
- solar_error
- wind_onshore_error
- wind_offshore_error

The error columns are computed after aligning timestamps and resampling all series to hourly resolution.

These error values are the main input for the statistical analysis in Phase 2,
where the uncertainty of each variable is studied before designing the battery dispatch model.

### Definition of forecast error

Forecast error is defined as:

error = actual − forecast

This definition was applied during the data pipeline stage when the merged
hourly dataset was created.

The merged files therefore already contain the following error columns:

- load_error
- solar_error
- wind_onshore_error
- wind_offshore_error

With this definition:

- Positive error means the real value was higher than forecast.
- Negative error means the real value was lower than forecast.

This convention is useful for later control design.

Example:

If solar_error < 0  
- The forecast expected more solar than actually produced  
- The system may need additional generation or battery discharge.

If load_error > 0  
- Real demand was higher than expected  
- The system must supply more power than planned.

In Phase 2, these error values are analyzed statistically in order to
understand the uncertainty that the controller must handle.

### Pipeline summary

Raw forecasts / actuals
- Timezone normalization
- Hourly resampling
- Variable renaming
- Merge by timestamp
- Error computation
- Processed dataset

## Phase 1 conclusions

Phase 1 produced a validated hourly dataset of forecasts, actuals, and forecast errors.

This stage established:
- Trustworthy variable selection
- Correct renewable forecast extraction
- Correct temporal alignment
- Consistent naming and merging

This dataset is the foundation for Phase 2, where forecast errors are studied statistically.